# Shoestring anime renders (Wan 2.2 14B, full quality)

**What to do:** Runtime > Change runtime type > pick the biggest GPU offered (A100, or L4 / G4), turn on High-RAM, then Runtime > **Run all**. Approve the Google Drive prompt. Leave the tab open until the last cell says DONE. Re-running skips clips that already exist.

Inputs come from `My Drive/Shoestring Colab/in/` (images + `jobs.json`). Outputs go to `My Drive/Shoestring Colab/out/`. Models: Wan 2.2 I2V-A14B (Apache-2.0), 8-bit GGUF experts from QuantStack (Apache-2.0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, subprocess
ROOT = '/content/drive/MyDrive/Shoestring Colab'
IN, OUT = f'{ROOT}/in', f'{ROOT}/out'
os.makedirs(OUT, exist_ok=True)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
jobs = json.load(open(f'{IN}/jobs.json'))
print(len(jobs), 'jobs')

In [ ]:
!pip -q install -U diffusers transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg gguf peft "huggingface_hub[hf_transfer]"


In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download, hf_hub_download
D14 = snapshot_download('Wan-AI/Wan2.2-I2V-A14B-Diffusers', allow_patterns=['model_index.json','scheduler/*','tokenizer/*','text_encoder/*','transformer/config.json','transformer_2/config.json','vae/*'])
HI = hf_hub_download('QuantStack/Wan2.2-I2V-A14B-GGUF', 'HighNoise/Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf')
LO = hf_hub_download('QuantStack/Wan2.2-I2V-A14B-GGUF', 'LowNoise/Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf')
print('models ready')

In [ ]:
import torch, time
from diffusers import WanImageToVideoPipeline, WanTransformer3DModel, GGUFQuantizationConfig
from diffusers.utils import export_to_video, load_image
q = GGUFQuantizationConfig(compute_dtype=torch.bfloat16)
hi = WanTransformer3DModel.from_single_file(HI, quantization_config=q, config=D14, subfolder='transformer', torch_dtype=torch.bfloat16)
lo = WanTransformer3DModel.from_single_file(LO, quantization_config=q, config=D14, subfolder='transformer_2', torch_dtype=torch.bfloat16)
pipe = WanImageToVideoPipeline.from_pretrained(D14, transformer=hi, transformer_2=lo, torch_dtype=torch.bfloat16)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram >= 70: pipe.to('cuda')
else: pipe.enable_model_cpu_offload()
print('pipeline ready on', round(vram), 'GB')

In [ ]:
NEG = ('text, letters, watermark, logo, subtitles, static, frozen frame, blurry, low quality, deformed hands, extra fingers, '
       'extra limbs, morphing face, flicker, 3d render, photorealistic, new characters appearing')
STYLE = '2D anime, cel-shaded, clean ink linework, consistent character, cinematic. '
log = open(f'{OUT}/timings.tsv', 'a')
for j in jobs:
    out = f"{OUT}/{j['id']}.mp4"
    if os.path.exists(out): print('skip', j['id']); continue
    t0 = time.time()
    img = load_image(f"{IN}/{j['image']}").resize((j.get('w',1280), j.get('h',720)))
    frames = pipe(image=img, prompt=STYLE + j['motion'], negative_prompt=NEG, height=j.get('h',720), width=j.get('w',1280),
                  num_frames=j.get('frames',81), guidance_scale=3.5, guidance_scale_2=3.5,
                  num_inference_steps=j.get('steps',30), generator=torch.Generator('cpu').manual_seed(j.get('seed',1))).frames[0]
    export_to_video(frames, out, fps=16)
    dt = int(time.time() - t0); log.write(f"{j['id']}\t{dt}\n"); log.flush()
    print(j['id'], dt, 's')
print('DONE')